In [14]:
import os
import io
import tarfile
import warnings

import urllib.request

import pandas as pd
import numpy as np

import astropy.units as u

from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astroquery.heasarc import Heasarc
from astropy.utils.exceptions import AstropyWarning

warnings.simplefilter('ignore', category=AstropyWarning)

heasarc = Heasarc()

clusters = ['A85', 'A426', 'A1644', 'A1656', 'A2029','A3158']

In [47]:
def retrieve(target):
    print('getting simbad coordinates for target object...\n')

    coord = SkyCoord.from_name(target)

    print('searching chandra catalog in given region...\n')

    results = heasarc.query_region(coord, catalog='chanmaster', radius=15 * u.arcmin)
    results.sort('exposure')
    results.reverse()

    print('results from chandra catalog in given region:\n')

    print(results['obsid', 'exposure', 'ra', 'dec'][:10])
    
    best_obsid = str(results['obsid'][0])

    base_url = f'https://heasarc.gsfc.nasa.gov/FTP/chandra/data/byobsid/{best_obsid[-1]}/{best_obsid}/primary/'

    filename_0 = f'acisf{best_obsid.zfill(5)}N000_evt2.fits.gz'
    filename_1 = f'acisf{best_obsid.zfill(5)}N001_evt2.fits.gz'
    filename_2 = f'acisf{best_obsid.zfill(5)}N002_evt2.fits.gz'
    filename_3 = f'acisf{best_obsid.zfill(5)}N003_evt2.fits.gz'
    filename_4 = f'acisf{best_obsid.zfill(5)}N004_evt2.fits.gz'
    filename_5 = f'acisf{best_obsid.zfill(5)}N005_evt2.fits.gz'
    filename_6 = f'acisf{best_obsid.zfill(5)}N006_evt2.fits.gz'
    filename_7 = f'acisf{best_obsid.zfill(5)}N007_evt2.fits.gz'
    filename_8 = f'acisf{best_obsid.zfill(5)}N008_evt2.fits.gz'
    filename_9 = f'acisf{best_obsid.zfill(5)}N009_evt2.fits.gz'

    download = True

    if os.path.exists('catalog/Chandra/' + filename_0):
        filename = filename_0
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_1):
        filename = filename_1
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_2):
        filename = filename_2
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_3):
        filename = filename_3
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_4):
        filename = filename_4
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_5):
        filename = filename_5
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_6):
        filename = filename_6
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_7):
        filename = filename_7
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_8):
        filename = filename_8
        print('\nmatching file found, skipping download.\n')
    elif os.path.exists('catalog/Chandra/' + filename_9):
        filename = filename_9
        print('\nmatching file found, skipping download.\n')
    else:
        try:
            print('\nattempting download...\n')
            filename = filename_0
            download_url = base_url + filename
            urllib.request.urlretrieve(full_download_url, 'catalog/Chandra/' + filename)
        except:
            try:
                print('file not found, trying fallback...\n')
                filename = filename_1
                download_url = base_url + filename
                urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
            except:
                try:
                    print('file not found, trying fallback...\n')
                    filename = filename_2
                    download_url = base_url + filename
                    urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                except:
                    try:
                        print('file not found, trying fallback...\n')
                        filename = filename_3
                        download_url = base_url + filename
                        urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                    except:
                        try:
                            print('file not found, trying fallback...\n')
                            filename = filename_4
                            download_url = base_url + filename
                            urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                        except:
                            try:
                                print('file not found, trying fallback...\n')
                                filename = filename_5
                                download_url = base_url + filename
                                urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                            except:
                                try:
                                    print('file not found, trying fallback...\n')
                                    filename = filename_6
                                    download_url = base_url + filename
                                    urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                                except:
                                    try:
                                        print('file not found, trying fallback...\n')
                                        filename = filename_7
                                        download_url = base_url + filename
                                        urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                                    except:
                                        try:
                                            print('file not found, trying fallback...\n')
                                            filename = filename_8
                                            download_url = base_url + filename
                                            urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                                        except:
                                            try:
                                                print('file not found, trying fallback...\n')
                                                filename = filename_9
                                                download_url = base_url + filename
                                                urllib.request.urlretrieve(download_url, 'catalog/Chandra/' + filename)
                                            except:
                                                download = False
                                                print('download failed, aborting...')

    if download:
        print('extracting binned spectrum...\n')
        with fits.open('catalog/Chandra/' + filename) as hdul:
            data = hdul[1].data
            header = hdul[1].header
        
            exposure = header.get('EXPOSURE', 1.0)
        
            energies = data['energy'] / 1000.0
        
            edges = np.linspace(2.0, 10.0, 501)
            counts, _ = np.histogram(energies, bins=edges)
        
            centers = (edges[:-1] + edges[1:]) / 2.0
            widths = np.diff(edges)
            
            flux = counts / (exposure * widths)
            error = np.sqrt(counts) / (exposure * widths)
        
            mask = flux > 0
        
            centers = centers[mask]
            flux = flux[mask]
            error = error[mask]
    
            output_file = f'{filename[:-8]}-{target}.txt'
            
            np.savetxt('catalog/Chandra/' + output_file, np.column_stack((centers, flux, error)), header='E / keV # rel_F # rel_F_err #')

        df = pd.read_csv('catalog/Chandra/ACCEPT.dat', sep=r'\s+', skiprows=[1])
        df.rename(columns={'#Name': 'Name'}, inplace=True)
        
        name = 'ABELL_' + target[1:].zfill(4)
        df = df[df['Name'] == name]
        
        if len(df) == 0:
            print('no matching profiles found.\n')
        else:
            print('saving profiles...\n')
            R = (df['Rin'] + df['Rout']) / 2.0
            ne = df['nelec']
            ne_err = df['neerr']
            K = df['Kitpl']
            K_err = df['Kerr']
            P = df['Pitpl']
            P_err = df['Perr']
            T = df['Tx']
            T_err = df['Txerr']

            output_file = f'{filename[:-8]}-{name}.dat'

            np.savetxt('catalog/Chandra/' + output_file,
                       np.column_stack((R, ne, ne_err, K, K_err, P, P_err, T, T_err)),
                       header='R / kpc # ne / cm-3 # ne_err / cm-3 # K / keV cm2 # K_err / keV cm2 # P / keV cm-3 # P_err / keV cm-3 # T / keV # T_err / keV #')

        print('done.')

In [45]:
def extract(target):
    data = {'density': None, 'pressure': None, 'temperature': None, 'entropy': None, 'abundance': None, 'fgas': None}

    print('streaming data from archive...\n')

    with tarfile.open('catalog/XMM-Newton/X-COP.tar.gz', 'r:gz') as tar:
        for member in tar.getmembers():
            if target in member.name and member.isfile() and member.name.endswith(".fits"):
                
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                if 'density' in member.name:
                    data['density'] = Table.read(byte_stream, format='fits')
                elif 'pressure' in member.name:
                    data['pressure'] = Table.read(byte_stream, format='fits')
                elif 'temperature' in member.name:
                    data['temperature'] = Table.read(byte_stream, format='fits')
                elif 'entropy' in member.name:
                    data['entropy'] = Table.read(byte_stream, format='fits')
                elif 'abund' in member.name:
                    data['abundance'] = Table.read(byte_stream, format='fits')
                elif 'fgas' in member.name:
                    data['fgas'] = Table.read(byte_stream, format='fits')

    if data['density'] is not None:
        tab = data['density']

        print('saving density data...\n')

        R = (tab['R_IN'] + tab['R_OUT']) / 2.0
        ne = tab['NE']
        ne_err_lo = np.abs(tab['NE'] - tab['NE_LOW'])
        ne_err_hi = np.abs(tab['NE'] - tab['NE_HIGH'])
    
        output_file = f'X-COP-{target}-dens.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, ne, ne_err_lo, ne_err_hi)),
                   header='R / kpc # ne / cm-3 # ne_err_lo / cm-3 # ne_err_hi / cm-3 #')
    else:
        print('no density data found.\n')
    
    if data['pressure'] is not None:
        tab = data['pressure']

        print('saving pressure data...\n')

        R = tab['RW_X']
        P = tab['P_X']
        P_err = tab['eP_X']
    
        output_file = f'X-COP-{target}-pres.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, P, P_err)),
                   header='R/R500 # P/P500 # P_err/P500 #')
    else:
        print('no pressure data found.\n')

    if data['temperature'] is not None:
        tab = data['temperature']

        print('saving temperature data...\n')
        
        R = tab['RW_X']
        T = tab['T_X']
        T_err = tab['eT_X']
    
        output_file = f'X-COP-{target}-temp.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, T, T_err)),
                   header='R/R500 # T/T500 # T_err/T500 #')
    else:
        print('no temperature data found.\n')

    if data['entropy'] is not None:
        tab = data['entropy']

        print('saving entropy data...\n')

        R = tab['RW_X']
        K = tab['K_X']
        K_err = tab['eK_X']
    
        output_file = f'X-COP-{target}-entr.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, K, K_err)),
                   header='R/R500 # K/K500 # K_err/K500 #')
    else:
        print('no entropy data found.\n')

    if data['abundance'] is not None:
        tab = data['abundance']

        print('saving abundance data...\n')

        R = tab['RADIUS']
        Z = tab['ZFE']
        Z_err = tab['ZFE_ERR']
    
        output_file = f'X-COP-{target}-abun.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, Z, Z_err)),
                   header='R/R500 # Z # Z_err #')
    else:
        print('no abundance data found.\n')
    

    if data['fgas'] is not None:
        tab = data['fgas']

        print('saving fgas data...\n')
    
        R = tab['RADIUS']
        fgas = tab['FGAS']
        fgas_err_lo = np.abs(tab['FGAS'] - tab['FGAS_LO'])
        fgas_err_hi = np.abs(tab['FGAS'] - tab['FGAS_HI'])
    
        output_file = f'X-COP-{target}-fgas.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, fgas, fgas_err_lo, fgas_err_hi)),
                   header='R/R500 # fgas # fgas_err_lo # fgas_err_hi #')
    else:
        print('no fgas data found.\n')

    print('finished.')

In [48]:
print('---------------------------------------------')
for target in clusters:
    retrieve(target)
    print('---------------------------------------------')

---------------------------------------------
getting simbad coordinates for target object...

searching chandra catalog in given region...

results from chandra catalog in given region:

obsid exposure    ra      dec   
         s       deg      deg   
----- -------- -------- --------
15173    43080 10.42482 -9.34768
15174    40080 10.47191 -9.40945
  904    38910 10.44083 -9.37917
16263    38660 10.47191 -9.40945
16264    37080 10.42482 -9.34768
28689    29910 10.44486 -9.11320
28329    20090 10.44486 -9.11320
28688    20000 10.44486 -9.11320
30563    17590 10.44486 -9.11320
28686    17180 10.44486 -9.11320

matching file found, skipping download.

extracting binned spectrum...

saving profiles...

done.
---------------------------------------------
getting simbad coordinates for target object...

searching chandra catalog in given region...

results from chandra catalog in given region:

obsid exposure    ra      dec   
         s       deg      deg   
----- -------- -------- ------

In [49]:
print('---------------------------------------------')
for target in clusters:
    extract(target)
    print('---------------------------------------------')

---------------------------------------------
streaming data from archive...

saving density data...

saving pressure data...

saving temperature data...

saving entropy data...

saving abundance data...

saving fgas data...

finished.
---------------------------------------------
streaming data from archive...

no density data found.

no pressure data found.

no temperature data found.

no entropy data found.

no abundance data found.

no fgas data found.

finished.
---------------------------------------------
streaming data from archive...

saving density data...

saving pressure data...

saving temperature data...

saving entropy data...

saving abundance data...

saving fgas data...

finished.
---------------------------------------------
streaming data from archive...

no density data found.

no pressure data found.

no temperature data found.

no entropy data found.

no abundance data found.

no fgas data found.

finished.
---------------------------------------------
streaming 

In [50]:
import io
import tarfile
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# Update this path to point to your data archive
tar_path = "chandra/allfiles.tar.gz"

def preview_and_load_tables(tar_filepath):
    """
    Scans the tar.gz archive for pre-computed profile tables,
    prints structural information, and returns valid parsed Astropy Tables.
    """
    extracted_tables = {}
    
    with tarfile.open(tar_filepath, "r:gz") as tar:
        for member in tar.getmembers():
            # Targets common pipeline names: profile.fits, sb_profile.dat, thermo_profiles.txt, etc.
            if any(k in member.name.lower() for k in ["profile", "density", "temperature", "sb_", "entropy", "abund", "pressure", "mass"]) and member.isfile():
                print(f"📦 Found precomputed data table: {member.name}")
                
                # Extract file contents directly into a byte buffer
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                try:
                    # Astropy dynamically handles FITS binary tables, CSVs, and space-separated ASCII
                    if member.name.endswith(".fits"):
                        t = Table.read(byte_stream, format="fits")
                    else:
                        t = Table.read(byte_stream, format="ascii")
                    
                    extracted_tables[member.name] = t
                    print(f"   ↳ Success. Columns found: {t.colnames}\n")
                except Exception as e:
                    print(f"   ↳ Could not automatically parse file layout: {e}\n")
                    
    return extracted_tables

# Execute search over the compressed archive
tables = preview_and_load_tables(tar_path)

# =====================================================================
# SIMULATION FALLBACK (Runs ONLY if your archive is missing the files)
# =====================================================================
if not tables:
    print("⚠️ No matching table filenames found. Generating a mock X-COP table for visualization.")
    mock_data = Table()
    mock_data['RADIUS'] = np.logspace(1, 3.2, 25) # Radius in arcsec / kpc
    mock_data['SB'] = 1e-2 * (1 + (mock_data['RADIUS']/150)**2)**(-3*0.62 + 0.5)
    mock_data['SB_ERR'] = mock_data['SB'] * 0.05
    mock_data['KT'] = 7.5 * (1 + 0.5 * (mock_data['RADIUS']/400)) / (1 + (mock_data['RADIUS']/350)**2)**0.25
    mock_data['KT_ERR'] = mock_data['KT'] * 0.07
    mock_data['DENSITY'] = 3e-3 * (1 + (mock_data['RADIUS']/180)**2)**(-1.5*0.65)
    mock_data['DENSITY_ERR'] = mock_data['DENSITY'] * 0.09
    tables['mock_xcop_profile.fits'] = mock_data

# =====================================================================
# PLOTTING FROM EXTRACTED DATA STRUCT
# =====================================================================
# Target the first loaded or simulated table
active_filename = list(tables.keys())[0]
t = tables[active_filename]

# Map X-COP standard column variations to consistent script keys
r_col = [c for c in t.colnames if c.upper() in ['R', 'RADIUS', 'R_KPC', 'R_ARCSEC']][0]
sb_col = [c for c in t.colnames if c.upper() in ['SB', 'SURF_BRIGHT', 'FLUX', 'INTENSITY']][0]
t_col = [c for c in t.colnames if c.upper() in ['T', 'KT', 'TEMP', 'TEMPERATURE']][0]
n_col = [c for c in t.colnames if c.upper() in ['NE', 'DENSITY', 'N_E', 'RHO']][0]

# Locate associated error columns (defaulting to zero if missing)
sb_err = [c for c in t.colnames if sb_col in c and 'ERR' in c.upper()] or None
t_err = [c for c in t.colnames if t_col in c and 'ERR' in c.upper()] or None
n_err = [c for c in t.colnames if n_col in c and 'ERR' in c.upper()] or None

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Surface Brightness
y_err_sb = t[sb_err[0]] if sb_err else None
axs[0].errorbar(t[r_col], t[sb_col], yerr=y_err_sb, fmt='o-', color='indigo', capsize=2)
axs[0].set_yscale('log')
axs[0].set_xscale('log')
axs[0].set_xlabel(f"Radius ({r_col})")
axs[0].set_ylabel("Surface Brightness")
axs[0].grid(True, which="both", ls="--", alpha=0.5)
axs[0].set_title("Surface Brightness Profile")

# Plot 2: Temperature Profile
y_err_t = t[t_err[0]] if t_err else None
axs[1].errorbar(t[r_col], t[t_col], yerr=y_err_t, fmt='s-', color='darkorange', capsize=2)
axs[1].set_xscale('log')
axs[1].set_xlabel(f"Radius ({r_col})")
axs[1].set_ylabel("kT [keV]")
axs[1].grid(True, which="both", ls="--", alpha=0.5)
axs[1].set_title("Temperature Profile")

# Plot 3: Density Profile
y_err_n = t[n_err[0]] if n_err else None
axs[2].errorbar(t[r_col], t[n_col], yerr=y_err_n, fmt='^-', color='teal', capsize=2)
axs[2].set_yscale('log')
axs[2].set_xscale('log')
axs[2].set_xlabel(f"Radius ({r_col})")
axs[2].set_ylabel("Density [cm^-3]")
axs[2].grid(True, which="both", ls="--", alpha=0.5)
axs[2].set_title("Electron Density Profile")

plt.suptitle(f"Profiles Plotted Straight from Memory: {active_filename}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

📦 Found precomputed data table: A1644/A1644_abund.fits
   ↳ Success. Columns found: ['RADIUS', 'WIDTH', 'ZFE', 'ZFE_ERR']

📦 Found precomputed data table: A1644/A1644_hydro_mass.fits
   ↳ Success. Columns found: ['RADIUS', 'M_FORW', 'EM_FORW', 'M_NFW', 'EM_NFW', 'M_EIN', 'EM_EIN', 'M_ISO', 'EM_ISO', 'M_BUR', 'EM_BUR', 'EM_HER']

📦 Found precomputed data table: A1644/A1644_temperature.fits
   ↳ Success. Columns found: ['RW_X', 'T_X', 'eT_X']

📦 Found precomputed data table: A1644/A1644_entropy.png
   ↳ Could not automatically parse file layout: 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte

📦 Found precomputed data table: A1644/A1644_entropy.fits
   ↳ Success. Columns found: ['RW_X', 'K_X', 'eK_X']

📦 Found precomputed data table: A1644/A1644_fgas_profile.fits
   ↳ Success. Columns found: ['RADIUS', 'M_NFW', 'M_NFW_LO', 'M_NFW_HI', 'MGAS', 'MGAS_LO', 'MGAS_HI', 'FGAS', 'FGAS_LO', 'FGAS_HI']

📦 Found precomputed data table: A1644/A1644_fgas_profile.png
   ↳ Could

IndexError: list index out of range

In [ ]:
import io
import tarfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

# Suppress the verbose FITS unit parsing warnings shown in your terminal logs
warnings.simplefilter('ignore', category=AstropyWarning)

# =====================================================================
# CONFIGURATION
# =====================================================================
tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A1644"  # Change to any available cluster name from your log

def extract_cluster_tables(tar_filepath, cluster_name):
    """
    Finds and reads the density, temperature, and fgas FITS tables
    for a specific cluster completely in memory.
    """
    data_dict = {'density': None, 'temperature': None, 'fgas': None}
    
    with tarfile.open(tar_filepath, "r:gz") as tar:
        for member in tar.getmembers():
            # Filter by the target cluster's directory/prefix name
            if cluster_name in member.name and member.isfile() and member.name.endswith(".fits"):
                
                # Isolate the file data stream
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                # Categorize based on file name identifiers
                if "density" in member.name:
                    data_dict['density'] = Table.read(byte_stream, format="fits")
                elif "temperature" in member.name:
                    data_dict['temperature'] = Table.read(byte_stream, format="fits")
                elif "fgas" in member.name:
                    data_dict['fgas'] = Table.read(byte_stream, format="fits")
                    
    return data_dict

# Extract the tables straight to variables
cluster_data = extract_cluster_tables(tar_path, target_cluster)

# Verify we successfully collected the necessary datasets
for key, table in cluster_data.items():
    if table is None:
        print(f"⚠️ Warning: Could not find the '{key}' table for cluster {target_cluster}.")
    else:
        print(f"✅ Successfully loaded {key} table ({len(table)} rows).")

# =====================================================================
# INTERACTIVE DATA PLOTTING
# =====================================================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"X-COP Science Profiles: Galaxy Cluster {target_cluster}", fontsize=15, y=1.05)

# --- PANEL 1: TEMPERATURE PROFILE ---
if cluster_data['temperature'] is not None:
    t_tab = cluster_data['temperature']
    # Radius column: 'RW_X', Temp: 'T_X', Error: 'eT_X'
    axs[0].errorbar(t_tab['RW_X'], t_tab['T_X'], yerr=t_tab['eT_X'], 
                    fmt='o-', color='crimson', markersize=6, capsize=3, label='X-ray Gas Temp')
    axs[0].set_xscale('log')
    axs[0].set_xlabel("Radius [$R / R_{500}$]")
    axs[0].set_ylabel("Temperature $kT$ [keV]")
    axs[0].set_title("Gas Temperature Profile")
    axs[0].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 2: ELECTRON DENSITY PROFILE ---
if cluster_data['density'] is not None:
    n_tab = cluster_data['density']
    # Radial midpoint calculation from R_IN and R_OUT
    r_mid = (n_tab['R_IN'] + n_tab['R_OUT']) / 2.0
    
    # Calculate directional error margins for asymmetric errorbars
    yerr_lower = n_tab['NE'] - n_tab['NE_LOW']
    yerr_upper = n_tab['NE_HIGH'] - n_tab['NE']
    
    axs[1].errorbar(r_mid, n_tab['NE'], yerr=[yerr_lower, yerr_upper], 
                    fmt='s-', color='teal', markersize=5, capsize=3, label='De-projected $n_e$')
    axs[1].set_xscale('log')
    axs[1].set_yscale('log')
    axs[1].set_xlabel("Radius [$R / R_{500}$]")
    axs[1].set_ylabel("Electron Density $n_e$ [$\mathrm{cm}^{-3}$]")
    axs[1].set_title("3D Electron Density Profile ($n_e$)")
    axs[1].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 3: GAS MASS FRACTION PROFILE ---
if cluster_data['fgas'] is not None:
    f_tab = cluster_data['fgas']
    f_err_lower = f_tab['FGAS'] - f_tab['FGAS_LO']
    f_err_upper = f_tab['FGAS_HI'] - f_tab['FGAS']
    
    axs[2].errorbar(f_tab['RADIUS'], f_tab['FGAS'], yerr=[f_err_lower, f_err_upper], 
                    fmt='^-', color='darkorchid', markersize=5, capsize=3, label='$f_{gas} = M_{gas}/M_{tot}$')
    axs[2].set_xscale('log')
    axs[2].set_xlabel("Radius [$R / R_{500}$]")
    axs[2].set_ylabel("Gas Mass Fraction $f_{gas}$")
    axs[2].set_title("Gas Mass Fraction Profile")
    axs[2].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import io
import tarfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

# Suppress the verbose FITS unit parsing warnings shown in your terminal logs
warnings.simplefilter('ignore', category=AstropyWarning)

# =====================================================================
# CONFIGURATION
# =====================================================================
tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A2029"  # Change to any available cluster name from your log

def extract_cluster_tables(tar_filepath, cluster_name):
    """
    Finds and reads the density, temperature, and fgas FITS tables
    for a specific cluster completely in memory.
    """
    data_dict = {'density': None, 'temperature': None, 'fgas': None}
    
    with tarfile.open(tar_filepath, "r:gz") as tar:
        for member in tar.getmembers():
            # Filter by the target cluster's directory/prefix name
            if cluster_name in member.name and member.isfile() and member.name.endswith(".fits"):
                
                # Isolate the file data stream
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                # Categorize based on file name identifiers
                if "density" in member.name:
                    data_dict['density'] = Table.read(byte_stream, format="fits")
                elif "temperature" in member.name:
                    data_dict['temperature'] = Table.read(byte_stream, format="fits")
                elif "fgas" in member.name:
                    data_dict['fgas'] = Table.read(byte_stream, format="fits")
                    
    return data_dict

# Extract the tables straight to variables
cluster_data = extract_cluster_tables(tar_path, target_cluster)

# Verify we successfully collected the necessary datasets
for key, table in cluster_data.items():
    if table is None:
        print(f"⚠️ Warning: Could not find the '{key}' table for cluster {target_cluster}.")
    else:
        print(f"✅ Successfully loaded {key} table ({len(table)} rows).")

# =====================================================================
# INTERACTIVE DATA PLOTTING
# =====================================================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"X-COP Science Profiles: Galaxy Cluster {target_cluster}", fontsize=15, y=1.05)

# --- PANEL 1: TEMPERATURE PROFILE ---
if cluster_data['temperature'] is not None:
    t_tab = cluster_data['temperature']
    # Radius column: 'RW_X', Temp: 'T_X', Error: 'eT_X'
    axs[0].errorbar(t_tab['RW_X'], t_tab['T_X'], yerr=t_tab['eT_X'], 
                    fmt='o-', color='crimson', markersize=6, capsize=3, label='X-ray Gas Temp')
    axs[0].set_xscale('log')
    axs[0].set_xlabel("Radius [$R / R_{500}$]")
    axs[0].set_ylabel("Temperature $kT$ [keV]")
    axs[0].set_title("Gas Temperature Profile")
    axs[0].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 2: ELECTRON DENSITY PROFILE ---
if cluster_data['density'] is not None:
    n_tab = cluster_data['density']
    # Radial midpoint calculation from R_IN and R_OUT
    r_mid = (n_tab['R_IN'] + n_tab['R_OUT']) / 2.0
    
    # Calculate directional error margins for asymmetric errorbars
    yerr_lower = n_tab['NE'] - n_tab['NE_LOW']
    yerr_upper = n_tab['NE_HIGH'] - n_tab['NE']
    
    axs[1].errorbar(r_mid, n_tab['NE'], yerr=[yerr_lower, yerr_upper], 
                    fmt='s-', color='teal', markersize=5, capsize=3, label='De-projected $n_e$')
    axs[1].set_xscale('log')
    axs[1].set_yscale('log')
    axs[1].set_xlabel("Radius [$R / R_{500}$]")
    axs[1].set_ylabel("Electron Density $n_e$ [$\mathrm{cm}^{-3}$]")
    axs[1].set_title("3D Electron Density Profile ($n_e$)")
    axs[1].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 3: GAS MASS FRACTION PROFILE ---
if cluster_data['fgas'] is not None:
    f_tab = cluster_data['fgas']
    f_err_lower = f_tab['FGAS'] - f_tab['FGAS_LO']
    f_err_upper = f_tab['FGAS_HI'] - f_tab['FGAS']
    
    axs[2].errorbar(f_tab['RADIUS'], f_tab['FGAS'], yerr=[f_err_lower, f_err_upper], 
                    fmt='^-', color='darkorchid', markersize=5, capsize=3, label='$f_{gas} = M_{gas}/M_{tot}$')
    axs[2].set_xscale('log')
    axs[2].set_xlabel("Radius [$R / R_{500}$]")
    axs[2].set_ylabel("Gas Mass Fraction $f_{gas}$")
    axs[2].set_title("Gas Mass Fraction Profile")
    axs[2].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import io
import tarfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

# Suppress the verbose FITS unit parsing warnings shown in your terminal logs
warnings.simplefilter('ignore', category=AstropyWarning)

# =====================================================================
# CONFIGURATION
# =====================================================================
tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A3158"  # Change to any available cluster name from your log

def extract_cluster_tables(tar_filepath, cluster_name):
    """
    Finds and reads the density, temperature, and fgas FITS tables
    for a specific cluster completely in memory.
    """
    data_dict = {'density': None, 'temperature': None, 'fgas': None}
    
    with tarfile.open(tar_filepath, "r:gz") as tar:
        for member in tar.getmembers():
            # Filter by the target cluster's directory/prefix name
            if cluster_name in member.name and member.isfile() and member.name.endswith(".fits"):
                
                # Isolate the file data stream
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                # Categorize based on file name identifiers
                if "density" in member.name:
                    data_dict['density'] = Table.read(byte_stream, format="fits")
                elif "temperature" in member.name:
                    data_dict['temperature'] = Table.read(byte_stream, format="fits")
                elif "fgas" in member.name:
                    data_dict['fgas'] = Table.read(byte_stream, format="fits")
                    
    return data_dict

# Extract the tables straight to variables
cluster_data = extract_cluster_tables(tar_path, target_cluster)

# Verify we successfully collected the necessary datasets
for key, table in cluster_data.items():
    if table is None:
        print(f"⚠️ Warning: Could not find the '{key}' table for cluster {target_cluster}.")
    else:
        print(f"✅ Successfully loaded {key} table ({len(table)} rows).")

# =====================================================================
# INTERACTIVE DATA PLOTTING
# =====================================================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"X-COP Science Profiles: Galaxy Cluster {target_cluster}", fontsize=15, y=1.05)

# --- PANEL 1: TEMPERATURE PROFILE ---
if cluster_data['temperature'] is not None:
    t_tab = cluster_data['temperature']
    # Radius column: 'RW_X', Temp: 'T_X', Error: 'eT_X'
    axs[0].errorbar(t_tab['RW_X'], t_tab['T_X'], yerr=t_tab['eT_X'], 
                    fmt='o-', color='crimson', markersize=6, capsize=3, label='X-ray Gas Temp')
    axs[0].set_xscale('log')
    axs[0].set_xlabel("Radius [$R / R_{500}$]")
    axs[0].set_ylabel("Temperature $kT$ [keV]")
    axs[0].set_title("Gas Temperature Profile")
    axs[0].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 2: ELECTRON DENSITY PROFILE ---
if cluster_data['density'] is not None:
    n_tab = cluster_data['density']
    # Radial midpoint calculation from R_IN and R_OUT
    r_mid = (n_tab['R_IN'] + n_tab['R_OUT']) / 2.0
    
    # Calculate directional error margins for asymmetric errorbars
    yerr_lower = n_tab['NE'] - n_tab['NE_LOW']
    yerr_upper = n_tab['NE_HIGH'] - n_tab['NE']
    
    axs[1].errorbar(r_mid, n_tab['NE'], yerr=[yerr_lower, yerr_upper], 
                    fmt='s-', color='teal', markersize=5, capsize=3, label='De-projected $n_e$')
    axs[1].set_xscale('log')
    axs[1].set_yscale('log')
    axs[1].set_xlabel("Radius [$R / R_{500}$]")
    axs[1].set_ylabel("Electron Density $n_e$ [$\mathrm{cm}^{-3}$]")
    axs[1].set_title("3D Electron Density Profile ($n_e$)")
    axs[1].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 3: GAS MASS FRACTION PROFILE ---
if cluster_data['fgas'] is not None:
    f_tab = cluster_data['fgas']
    f_err_lower = f_tab['FGAS'] - f_tab['FGAS_LO']
    f_err_upper = f_tab['FGAS_HI'] - f_tab['FGAS']
    
    axs[2].errorbar(f_tab['RADIUS'], f_tab['FGAS'], yerr=[f_err_lower, f_err_upper], 
                    fmt='^-', color='darkorchid', markersize=5, capsize=3, label='$f_{gas} = M_{gas}/M_{tot}$')
    axs[2].set_xscale('log')
    axs[2].set_xlabel("Radius [$R / R_{500}$]")
    axs[2].set_ylabel("Gas Mass Fraction $f_{gas}$")
    axs[2].set_title("Gas Mass Fraction Profile")
    axs[2].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import io
import tarfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

# Suppress the verbose FITS unit parsing warnings shown in your terminal logs
warnings.simplefilter('ignore', category=AstropyWarning)

# =====================================================================
# CONFIGURATION
# =====================================================================
tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A644"  # Change to any available cluster name from your log

def extract_cluster_tables(tar_filepath, cluster_name):
    """
    Finds and reads the density, temperature, and fgas FITS tables
    for a specific cluster completely in memory.
    """
    data_dict = {'density': None, 'temperature': None, 'fgas': None}
    
    with tarfile.open(tar_filepath, "r:gz") as tar:
        for member in tar.getmembers():
            # Filter by the target cluster's directory/prefix name
            if cluster_name in member.name and member.isfile() and member.name.endswith(".fits"):
                
                # Isolate the file data stream
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                # Categorize based on file name identifiers
                if "density" in member.name:
                    data_dict['density'] = Table.read(byte_stream, format="fits")
                elif "temperature" in member.name:
                    data_dict['temperature'] = Table.read(byte_stream, format="fits")
                elif "fgas" in member.name:
                    data_dict['fgas'] = Table.read(byte_stream, format="fits")
                    
    return data_dict

# Extract the tables straight to variables
cluster_data = extract_cluster_tables(tar_path, target_cluster)

# Verify we successfully collected the necessary datasets
for key, table in cluster_data.items():
    if table is None:
        print(f"⚠️ Warning: Could not find the '{key}' table for cluster {target_cluster}.")
    else:
        print(f"✅ Successfully loaded {key} table ({len(table)} rows).")

# =====================================================================
# INTERACTIVE DATA PLOTTING
# =====================================================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"X-COP Science Profiles: Galaxy Cluster {target_cluster}", fontsize=15, y=1.05)

# --- PANEL 1: TEMPERATURE PROFILE ---
if cluster_data['temperature'] is not None:
    t_tab = cluster_data['temperature']
    # Radius column: 'RW_X', Temp: 'T_X', Error: 'eT_X'
    axs[0].errorbar(t_tab['RW_X'], t_tab['T_X'], yerr=t_tab['eT_X'], 
                    fmt='o-', color='crimson', markersize=6, capsize=3, label='X-ray Gas Temp')
    axs[0].set_xscale('log')
    axs[0].set_xlabel("Radius [$R / R_{500}$]")
    axs[0].set_ylabel("Temperature $kT$ [keV]")
    axs[0].set_title("Gas Temperature Profile")
    axs[0].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 2: ELECTRON DENSITY PROFILE ---
if cluster_data['density'] is not None:
    n_tab = cluster_data['density']
    # Radial midpoint calculation from R_IN and R_OUT
    r_mid = (n_tab['R_IN'] + n_tab['R_OUT']) / 2.0
    
    # Calculate directional error margins for asymmetric errorbars
    yerr_lower = n_tab['NE'] - n_tab['NE_LOW']
    yerr_upper = n_tab['NE_HIGH'] - n_tab['NE']
    
    axs[1].errorbar(r_mid, n_tab['NE'], yerr=[yerr_lower, yerr_upper], 
                    fmt='s-', color='teal', markersize=5, capsize=3, label='De-projected $n_e$')
    axs[1].set_xscale('log')
    axs[1].set_yscale('log')
    axs[1].set_xlabel("Radius [$R / R_{500}$]")
    axs[1].set_ylabel("Electron Density $n_e$ [$\mathrm{cm}^{-3}$]")
    axs[1].set_title("3D Electron Density Profile ($n_e$)")
    axs[1].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 3: GAS MASS FRACTION PROFILE ---
if cluster_data['fgas'] is not None:
    f_tab = cluster_data['fgas']
    f_err_lower = f_tab['FGAS'] - f_tab['FGAS_LO']
    f_err_upper = f_tab['FGAS_HI'] - f_tab['FGAS']
    
    axs[2].errorbar(f_tab['RADIUS'], f_tab['FGAS'], yerr=[f_err_lower, f_err_upper], 
                    fmt='^-', color='darkorchid', markersize=5, capsize=3, label='$f_{gas} = M_{gas}/M_{tot}$')
    axs[2].set_xscale('log')
    axs[2].set_xlabel("Radius [$R / R_{500}$]")
    axs[2].set_ylabel("Gas Mass Fraction $f_{gas}$")
    axs[2].set_title("Gas Mass Fraction Profile")
    axs[2].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import io
import tarfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

# Suppress the verbose FITS unit parsing warnings shown in your terminal logs
warnings.simplefilter('ignore', category=AstropyWarning)

# =====================================================================
# CONFIGURATION
# =====================================================================
tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A85"  # Change to any available cluster name from your log

def extract_cluster_tables(tar_filepath, cluster_name):
    """
    Finds and reads the density, temperature, and fgas FITS tables
    for a specific cluster completely in memory.
    """
    data_dict = {'density': None, 'temperature': None, 'fgas': None}
    
    with tarfile.open(tar_filepath, "r:gz") as tar:
        for member in tar.getmembers():
            # Filter by the target cluster's directory/prefix name
            if cluster_name in member.name and member.isfile() and member.name.endswith(".fits"):
                
                # Isolate the file data stream
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                # Categorize based on file name identifiers
                if "density" in member.name:
                    data_dict['density'] = Table.read(byte_stream, format="fits")
                elif "temperature" in member.name:
                    data_dict['temperature'] = Table.read(byte_stream, format="fits")
                elif "fgas" in member.name:
                    data_dict['fgas'] = Table.read(byte_stream, format="fits")
                    
    return data_dict

# Extract the tables straight to variables
cluster_data = extract_cluster_tables(tar_path, target_cluster)

# Verify we successfully collected the necessary datasets
for key, table in cluster_data.items():
    if table is None:
        print(f"⚠️ Warning: Could not find the '{key}' table for cluster {target_cluster}.")
    else:
        print(f"✅ Successfully loaded {key} table ({len(table)} rows).")

# =====================================================================
# INTERACTIVE DATA PLOTTING
# =====================================================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"X-COP Science Profiles: Galaxy Cluster {target_cluster}", fontsize=15, y=1.05)

# --- PANEL 1: TEMPERATURE PROFILE ---
if cluster_data['temperature'] is not None:
    t_tab = cluster_data['temperature']
    # Radius column: 'RW_X', Temp: 'T_X', Error: 'eT_X'
    axs[0].errorbar(t_tab['RW_X'], t_tab['T_X'], yerr=t_tab['eT_X'], 
                    fmt='o-', color='crimson', markersize=6, capsize=3, label='X-ray Gas Temp')
    axs[0].set_xscale('log')
    axs[0].set_xlabel("Radius [$R / R_{500}$]")
    axs[0].set_ylabel("Temperature $kT$ [keV]")
    axs[0].set_title("Gas Temperature Profile")
    axs[0].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 2: ELECTRON DENSITY PROFILE ---
if cluster_data['density'] is not None:
    n_tab = cluster_data['density']
    # Radial midpoint calculation from R_IN and R_OUT
    r_mid = (n_tab['R_IN'] + n_tab['R_OUT']) / 2.0
    
    # Calculate directional error margins for asymmetric errorbars
    yerr_lower = n_tab['NE'] - n_tab['NE_LOW']
    yerr_upper = n_tab['NE_HIGH'] - n_tab['NE']
    
    axs[1].errorbar(r_mid, n_tab['NE'], yerr=[yerr_lower, yerr_upper], 
                    fmt='s-', color='teal', markersize=5, capsize=3, label='De-projected $n_e$')
    axs[1].set_xscale('log')
    axs[1].set_yscale('log')
    axs[1].set_xlabel("Radius [$R / R_{500}$]")
    axs[1].set_ylabel("Electron Density $n_e$ [$\mathrm{cm}^{-3}$]")
    axs[1].set_title("3D Electron Density Profile ($n_e$)")
    axs[1].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 3: GAS MASS FRACTION PROFILE ---
if cluster_data['fgas'] is not None:
    f_tab = cluster_data['fgas']
    f_err_lower = f_tab['FGAS'] - f_tab['FGAS_LO']
    f_err_upper = f_tab['FGAS_HI'] - f_tab['FGAS']
    
    axs[2].errorbar(f_tab['RADIUS'], f_tab['FGAS'], yerr=[f_err_lower, f_err_upper], 
                    fmt='^-', color='darkorchid', markersize=5, capsize=3, label='$f_{gas} = M_{gas}/M_{tot}$')
    axs[2].set_xscale('log')
    axs[2].set_xlabel("Radius [$R / R_{500}$]")
    axs[2].set_ylabel("Gas Mass Fraction $f_{gas}$")
    axs[2].set_title("Gas Mass Fraction Profile")
    axs[2].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import io
import tarfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

# Suppress the verbose FITS unit parsing warnings shown in your terminal logs
warnings.simplefilter('ignore', category=AstropyWarning)

# =====================================================================
# CONFIGURATION
# =====================================================================
tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A85"  # Change to any available cluster name from your log

def extract_cluster_tables(target):
    data_dict = {'density': None, 'temperature': None, 'fgas': None}
    
    with tarfile.open('catalog/XMM-Newton/X-COP.tar.gz', 'r:gz') as tar:
        for member in tar.getmembers():
            if target in member.name and member.isfile() and member.name.endswith(".fits"):
                
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                if 'density' in member.name:
                    data_dict['density'] = Table.read(byte_stream, format='fits')
                elif 'temperature' in member.name:
                    data_dict['temperature'] = Table.read(byte_stream, format='fits')
                elif 'fgas' in member.name:
                    data_dict['fgas'] = Table.read(byte_stream, format='fits')
                    
    return data_dict

# Extract the tables straight to variables
cluster_data = extract_cluster_tables(tar_path, target_cluster)

# Verify we successfully collected the necessary datasets
for key, table in cluster_data.items():
    if table is None:
        print(f"⚠️ Warning: Could not find the '{key}' table for cluster {target_cluster}.")
    else:
        print(f"✅ Successfully loaded {key} table ({len(table)} rows).")

# =====================================================================
# INTERACTIVE DATA PLOTTING
# =====================================================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"X-COP Science Profiles: Galaxy Cluster {target_cluster}", fontsize=15, y=1.05)

# --- PANEL 1: TEMPERATURE PROFILE ---
if cluster_data['temperature'] is not None:
    t_tab = cluster_data['temperature']
    # Radius column: 'RW_X', Temp: 'T_X', Error: 'eT_X'
    axs[0].errorbar(t_tab['RW_X'], t_tab['T_X'], yerr=t_tab['eT_X'], 
                    fmt='o-', color='crimson', markersize=6, capsize=3, label='X-ray Gas Temp')
    axs[0].set_xscale('log')
    axs[0].set_xlabel("Radius [$R / R_{500}$]")
    axs[0].set_ylabel("Temperature $kT$ [keV]")
    axs[0].set_title("Gas Temperature Profile")
    axs[0].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 2: ELECTRON DENSITY PROFILE ---
if cluster_data['density'] is not None:
    n_tab = cluster_data['density']
    # Radial midpoint calculation from R_IN and R_OUT
    r_mid = (n_tab['R_IN'] + n_tab['R_OUT']) / 2.0
    
    # Calculate directional error margins for asymmetric errorbars
    yerr_lower = n_tab['NE'] - n_tab['NE_LOW']
    yerr_upper = n_tab['NE_HIGH'] - n_tab['NE']
    
    axs[1].errorbar(r_mid, n_tab['NE'], yerr=[yerr_lower, yerr_upper], 
                    fmt='s-', color='teal', markersize=5, capsize=3, label='De-projected $n_e$')
    axs[1].set_xscale('log')
    axs[1].set_yscale('log')
    axs[1].set_xlabel("Radius [$R / R_{500}$]")
    axs[1].set_ylabel("Electron Density $n_e$ [$\mathrm{cm}^{-3}$]")
    axs[1].set_title("3D Electron Density Profile ($n_e$)")
    axs[1].grid(True, which="both", ls="--", alpha=0.4)

# --- PANEL 3: GAS MASS FRACTION PROFILE ---
if cluster_data['fgas'] is not None:
    f_tab = cluster_data['fgas']
    f_err_lower = f_tab['FGAS'] - f_tab['FGAS_LO']
    f_err_upper = f_tab['FGAS_HI'] - f_tab['FGAS']
    
    axs[2].errorbar(f_tab['RADIUS'], f_tab['FGAS'], yerr=[f_err_lower, f_err_upper], 
                    fmt='^-', color='darkorchid', markersize=5, capsize=3, label='$f_{gas} = M_{gas}/M_{tot}$')
    axs[2].set_xscale('log')
    axs[2].set_xlabel("Radius [$R / R_{500}$]")
    axs[2].set_ylabel("Gas Mass Fraction $f_{gas}$")
    axs[2].set_title("Gas Mass Fraction Profile")
    axs[2].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
def extract(target):
    data = {'density': None, 'temperature': None, 'fgas': None}

    print('streaming data from archive...\n')

    with tarfile.open('catalog/XMM-Newton/X-COP.tar.gz', 'r:gz') as tar:
        for member in tar.getmembers():
            if target in member.name and member.isfile() and member.name.endswith(".fits"):
                
                file_bytes = tar.extractfile(member).read()
                byte_stream = io.BytesIO(file_bytes)
                
                if 'density' in member.name:
                    data['density'] = Table.read(byte_stream, format='fits')
                elif 'temperature' in member.name:
                    data['temperature'] = Table.read(byte_stream, format='fits')
                elif 'fgas' in member.name:
                    data['fgas'] = Table.read(byte_stream, format='fits')

    if data['density'] is not None:
        tab = data['density']

        print('saving density data...\n')

        R = (tab['R_IN'] + tab['R_OUT']) / 2.0
        ne = tab['NE']
        ne_err_lo = np.abs(tab['NE'] - tab['NE_LOW'])
        ne_err_hi = np.abs(tab['NE'] - tab['NE_HIGH'])
    
        output_file = f'X-COP-{target}-dens.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, ne, ne_err_lo, ne_err_hi)),
                   header='R / kpc # ne / cm-3 # ne_err_lo / cm-3 # ne_err_hi / cm-3 #')
    else:
        print('no density data found...\n')

    if data['temperature'] is not None:
        tab = data['temperature']

        print('saving temperature data...\n')
        
        R = tab['RW_X']
        T = tab['T_X']
        T_err = tab['eT_X']
    
        output_file = f'X-COP-{target}-temp.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, T, T_err)),
                   header='R/R500 # T/Tavg # T_err/Tavg #')
    else:
        print('no temperature data found...\n')

    if data['fgas'] is not None:
        tab = data['fgas']

        print('saving fgas data...\n')
    
        R = tab['RADIUS']
        fgas = tab['FGAS']
        fgas_err_lo = np.abs(tab['FGAS'] - tab['FGAS_LO'])
        fgas_err_hi = np.abs(tab['FGAS'] - tab['FGAS_HI'])
    
        output_file = f'X-COP-{target}-fgas.txt'
    
        np.savetxt('catalog/XMM-Newton/' + output_file,
                   np.column_stack((R, fgas, fgas_err_lo, fgas_err_hi)),
                   header='R/R500 # fgas # fgas_err_lo # fgas_err_hi #')
    else:
        print('no fgas data found...\n')

    print('finished.')

In [ ]:
extract('A85')

In [10]:
import io
import tarfile
from astropy.io import fits

tar_path = "chandra/allfiles.tar.gz"
target_cluster = "A2029"  # Change to whichever cluster you want to inspect

with tarfile.open(tar_path, "r:gz") as tar:
    for member in tar.getmembers():
        # Look for the FITS tables matching our target cluster
        if target_cluster in member.name and member.name.endswith(".fits"):
            
            # Read the file directly from the compressed stream into memory
            file_bytes = tar.extractfile(member).read()
            byte_stream = io.BytesIO(file_bytes)
            
            # Open the FITS file with astropy
            with fits.open(byte_stream) as hdul:
                print(f"\n==================================================")
                print(f"FILE: {member.name}")
                print(f"==================================================")
                
                # HDU 0 or 1 contains the structural metadata header
                # We'll check both the primary header and the data table header
                for i, hdu in enumerate(hdul):
                    header = hdu.header
                    
                    # Print standard metadata keywords if they exist in this file
                    # X-COP commonly uses variants of these keys:
                    keywords_to_check = [
                        'R500', 'R_500', 'M500', 'M_500', 'T_AVG', 'T_500', 
                        'REDSHIFT', 'Z', 'KPC_ARCS', 'SCALE', 'TMAX'
                    ]
                    
                    found_any = False
                    for key in keywords_to_check:
                        if key in header:
                            print(f"  {key:<12} = {header[key]} \t# {header.comments[key]}")
                            found_any = True
                            
                    # If you want to dump the ENTIRE structural header to see 
                    # exactly what hidden constants the pipeline left behind:
                    # print(header.tostring(sep='\n'))
                    
                    if found_any:
                        break # Found the main metadata block, move to next file


FILE: A2029/A2029_pressure.fits
  R500         = 1414.0 	# Hydrostatic-equilibrium R500 in kpc
  M500         = 8.6511 	# Hydrostatic-equilibrium M500 in 1e14 Msun
  REDSHIFT     = 0.0766 	# Redshift

FILE: A2029/A2029_fgas_profile.fits
  R500         = 1414.0 	# NFW R500
  M500         = 8.6511 	# Hydrostatic-equilibrium M500 in 1e14 Msun
  REDSHIFT     = 0.0766 	# Redshift

FILE: A2029/A2029_hydro_mass.fits
  R500         = 1414.0 	# NFW R500
  M500         = 8.6511 	# NFW M500 in 1e14 Msun
  REDSHIFT     = 0.0766 	# Redshift

FILE: A2029/spectral_results_A2029.fits
  REDSHIFT     = 0.0766 	# 

FILE: A2029/A2029_mstar.fits
  R500         = 1414.0 	# NFW R500
  M500         = 8.6511 	# NFW M500 in 1e14 Msun
  REDSHIFT     = 0.0766 	# Redshift

FILE: A2029/A2029_entropy.fits
  R500         = 1414.0 	# Hydrostatic-equilibrium R500 in kpc
  M500         = 8.6511 	# Hydrostatic-equilibrium M500 in 1e14 Msun
  REDSHIFT     = 0.0766 	# Redshift

FILE: A2029/A2029_abund.fits
  R500         

,Name,Rin,Rout,nelec,neerr,Kitpl,Kflat,Kerr,Pitpl,Pflat,Perr,Mgrav,Merr,Tx,Txerr,Lambda,tcool5/2,t52err,tcool3/2,t32err
0,1E0657_56,1.17350,1.19450,0.000286,0.000020,2267.200,2267.200,631.2300,4.552000e-12,4.552000e-12,1.289200e-12,6.471500e+15,1.302800e+15,10.1160,2.91780,2.668200e-23,387.3800,114.95000,232.4300,68.97200
1,1E0657_56,1.15260,1.17350,0.000372,0.000042,1858.200,1858.200,507.2800,5.788400e-12,5.788400e-12,1.655100e-12,-4.372400e+14,-8.406300e+13,9.8517,2.70810,2.626500e-23,294.4800,87.65000,176.6900,52.59000
2,1E0657_56,1.13160,1.15260,0.000365,0.000044,1837.600,1837.600,481.2800,5.546700e-12,5.546700e-12,1.536200e-12,-1.631600e+15,-2.981900e+14,9.5879,2.49850,2.584800e-23,296.7600,85.23800,178.0600,51.14300
3,1E0657_56,1.11070,1.13160,0.000339,0.000045,1885.000,1885.000,474.6800,5.023000e-12,5.023000e-12,1.359600e-12,1.338200e+15,2.312600e+14,9.3241,2.28880,2.543100e-23,316.1200,88.28100,189.6700,52.96800
4,1E0657_56,1.08970,1.11070,0.000453,0.000065,1514.100,1514.100,364.9800,6.555700e-12,6.555700e-12,1.728100e-12,1.041300e+15,1.690600e+14,9.0603,2.07920,2.501400e-23,233.3800,63.11500,140.0300,37.86900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11159,ZWICKY_2701,0.06777,0.08471,0.005558,0.000410,160.600,160.600,22.7660,4.537200e-11,4.537200e-11,6.898200e-12,1.771300e+13,1.727100e+12,4.9952,0.64899,2.024900e-23,12.9570,1.93570,7.7744,1.16140
11160,ZWICKY_2701,0.05083,0.06777,0.010082,0.000386,98.837,98.837,12.4570,7.534100e-11,7.534100e-11,9.736000e-12,1.166800e+13,1.056000e+12,4.7821,0.61122,2.024100e-23,6.8411,0.91282,4.1047,0.54769
11161,ZWICKY_2701,0.03388,0.05083,0.013649,0.000324,70.493,70.493,7.4957,8.901800e-11,8.901800e-11,9.595400e-12,7.586000e+12,5.849500e+11,4.0584,0.43209,1.948400e-23,4.4554,0.48598,2.6732,0.29159
11162,ZWICKY_2701,0.01694,0.03388,0.016371,0.000372,54.812,54.812,4.5230,9.372800e-11,9.372800e-11,7.895500e-12,1.860000e+12,1.106400e+11,3.6982,0.33227,2.013300e-23,3.2755,0.30356,1.9653,0.18214


In [44]:
df = pd.read_csv('catalog/Chandra/ACCEPT.dat', sep=r'\s+', skiprows=[1])
df.rename(columns={'#Name': 'Name'}, inplace=True)

string = 'A3158'
name = 'ABELL_' + string[1:].zfill(4)
df = df[df['Name'] == name]

if len(df) == 0:
    print('no profiles found.\n')
else:
    print('saving profiles...\n')
    R = (df['Rin'] + df['Rout']) / 2.0
    ne = df['nelec']
    ne_err = df['neerr']
    K = df['Kitpl']
    K_err = df['Kerr']
    P = df['Pitpl']
    P_err = df['Perr']
    T = df['Tx']
    T_err = df['Txerr']

    

df

,Name,Rin,Rout,nelec,neerr,Kitpl,Kflat,Kerr,Pitpl,Pflat,Perr,Mgrav,Merr,Tx,Txerr,Lambda,tcool5/2,t52err,tcool3/2,t32err
5886,ABELL_3158,0.47588,0.47754,0.002625,0.000021,224.07,224.07,15.703,1.812900e-11,1.812900e-11,1.275000e-12,-1.560500e+16,-7.997400e+14,4.1500,0.26212,1.821800e-23,25.337,1.6128,15.2020,0.96766
5887,ABELL_3158,0.47034,0.47588,0.001276,0.000019,362.95,362.95,25.749,8.824700e-12,8.824700e-12,6.338700e-13,-5.763900e+14,-2.968500e+13,4.1655,0.26701,1.821700e-23,52.319,3.4451,31.3920,2.06710
5888,ABELL_3158,0.46481,0.47034,0.001166,0.000019,387.11,387.11,27.955,8.105500e-12,8.105500e-12,5.937600e-13,-3.522500e+15,-1.843700e+14,4.1894,0.27453,1.821700e-23,57.558,3.8894,34.5350,2.33370
5889,ABELL_3158,0.45928,0.46481,0.001104,0.000021,403.35,403.35,29.683,7.709800e-12,7.709800e-12,5.776700e-13,-5.087200e+14,-2.704900e+13,4.2131,0.28205,1.821600e-23,61.142,4.2530,36.6850,2.55180
5890,ABELL_3158,0.45374,0.45928,0.001022,0.000024,426.61,426.61,32.113,7.170500e-12,7.170500e-12,5.541700e-13,-3.085800e+14,-1.666200e+13,4.2370,0.28957,1.821500e-23,66.422,4.8002,39.8530,2.88010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5968,ABELL_3158,0.02213,0.02767,0.006284,0.000252,182.33,182.33,17.319,6.321200e-11,6.321200e-11,6.295000e-12,1.154700e+12,7.718100e+10,5.8902,0.47114,2.264200e-23,12.085,1.0814,7.2509,0.64885
5969,ABELL_3158,0.01660,0.02213,0.006481,0.000326,181.34,181.34,18.140,6.619000e-11,6.619000e-11,7.070800e-12,5.172200e+11,3.574800e+10,5.9522,0.48958,2.279000e-23,11.764,1.1341,7.0584,0.68049
5970,ABELL_3158,0.01107,0.01660,0.006598,0.000450,181.89,181.89,19.525,6.839400e-11,6.839400e-11,8.121700e-12,1.017000e+11,7.253000e+09,6.0142,0.50800,2.293800e-23,11.601,1.2590,6.9607,0.75539
5971,ABELL_3158,0.00553,0.01107,0.006184,0.000678,192.74,192.74,23.897,6.505100e-11,6.505100e-11,9.658000e-12,1.182000e+11,8.683200e+09,6.0762,0.52643,2.308500e-23,12.426,1.7357,7.4554,1.04140
